# Project Performance Analysis

This notebook performs exploratory data analysis (EDA) and builds predictive models on a synthetic dataset of project performance. The goal is to simulate tasks a business analyst or program manager might perform when evaluating project outcomes.

The dataset includes the following columns:

- `project_id`: Unique identifier for each project
- `project_name`: Name of the project
- `start_date`: Project start date
- `end_date`: Project end date
- `budget`: Allocated budget for the project (USD)
- `actual_spend`: Actual amount spent on the project (USD)
- `team_size`: Number of team members
- `risk_score`: Risk score between 0 and 1
- `status`: Current status of the project (`Completed`, `On Track`, `Delayed`, `Cancelled`, `At Risk`)
- `success`: Binary indicator (1 if the project was successful, 0 otherwise)

We'll explore the dataset, visualize key variables, and train a predictive model to estimate the likelihood of project success.


In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

# Display settings
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')


In [ ]:
# Load the dataset
df = pd.read_csv('project_data.csv')

# Show first few rows
df.head()


In [ ]:
# Summary statistics
df.describe(include='all')


In [ ]:
# Visualize project status distribution
plt.figure(figsize=(8,5))
sns.countplot(data=df, x='status', order=df['status'].value_counts().index)
plt.title('Project Status Distribution')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# Visualize budget vs. actual spend
plt.figure(figsize=(8,6))
sns.scatterplot(data=df, x='budget', y='actual_spend', hue='success')
plt.plot([df['budget'].min(), df['budget'].max()], [df['budget'].min(), df['budget'].max()], color='red', linestyle='--', label='Budget = Actual')
plt.title('Budget vs Actual Spend')
plt.legend()
plt.show()


In [ ]:
# Correlation heatmap
# Convert dates to numeric durations for correlation
df['duration'] = (pd.to_datetime(df['end_date']) - pd.to_datetime(df['start_date'])).dt.days

# Select numeric features
numeric_cols = ['budget', 'actual_spend', 'team_size', 'risk_score', 'duration', 'success']
correlation = df[numeric_cols].corr()

plt.figure(figsize=(8,6))
sns.heatmap(correlation, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix')
plt.show()


In [ ]:
# Prepare data for modeling
# Features: numeric columns except success
X = df[['budget', 'actual_spend', 'team_size', 'risk_score', 'duration']]
y = df['success']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Standardize numeric features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train logistic regression model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

# Predictions
y_pred = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)[:,1]


In [ ]:
# Evaluation metrics
print('Classification Report:')
print(classification_report(y_test, y_pred))
print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred))
print('ROC AUC Score:', roc_auc_score(y_test, y_proba))

# Plot ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label='Logistic Regression (AUC = {:.2f})'.format(roc_auc_score(y_test, y_proba)))
plt.plot([0,1],[0,1],'--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()
